# byte pair encoding


In [ ]:
import tiktoken
print("version:", tiktoken.__version__)     

version: 0.14.0


Once installed, we can instantiate the BPE tokenizer from tiktoken as follows:

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [4]:
string = tokenizer.decode(integers)
print(string)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [5]:
print("vocab size:", tokenizer.n_vocab)

vocab size: 50257


lets load our dataset and apply and check how many token does it have


In [6]:
from datasets import load_from_disk

dataset = load_from_disk(
    r"C:\LLM from Scratch\datasets\tiny_stories_dataset"
)

c:\LLM from Scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [8]:
dataset_text = " ".join(dataset["train"]['text'][:20000]) 

In [9]:
encode_dataset = tokenizer.encode(dataset_text, allowed_special={"<|endoftext|>"})

In [10]:
print("total tokens in the dataset:", len(encode_dataset))

total tokens in the dataset: 4443976


### How Embedding Models Learn via Next-Word Prediction

Embedding models use **Self-Supervised Learning** to train directly on raw text without human labels:

1. **Input-Target Pairs:** A sliding window splits text into an **Input** (a sequence of words) and a **Target** (the immediate next word).
2. **Prediction:** The model uses its current word embeddings to predict the Target.
3. **Error Calculation:** A loss function measures how far the prediction was from the actual Target.
4. **Weight Update:** Through backpropagation, the model adjusts its embedding weights to make a better prediction next time.

**The Result:** By repeatedly guessing the next word and correcting its errors, the model naturally updates its embedding weights so that words sharing similar contexts end up with similar mathematical representations.

In [13]:
encode_sample = encode_dataset[100:]

One of the easiest and most intuitive ways to create the input–target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input
tokens and y contains the targets, which are the inputs shifted by 1

In [16]:
context_size = 4
x = encode_sample[:context_size]
y = encode_sample[1:context_size + 1]
print("x:", x)
print("y:      ", y)

x: [198, 198, 41631, 11]
y:       [198, 41631, 11, 484]


In [18]:
for i in range(1,context_size+1):
    context = encode_sample[:i]
    desired = encode_sample[i]
    print(context, "->", desired)

[198] -> 198
[198, 198] -> 41631
[198, 198, 41631] -> 11
[198, 198, 41631, 11] -> 484


Everything left of the arrow (---->) refers to the input an LLM would receive, and
the token ID on the right side of the arrow represents the target token ID that the
LLM is supposed to predict. Let’s repeat the previous code but convert the token IDs
into tex

In [20]:
for i in range(0,context_size+1):
    context = encode_sample[i:i+context_size]
    desired = encode_sample[i+context_size]
    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))



Together, ----->  they

Together, they ----->  shared
Together, they shared ----->  the
, they shared the ----->  needle
 they shared the needle ----->  and


### Creating Input & Target Sequences

This code converts tokenized text into **training samples for a language model**.

* `input_ids` → tokens given to the model.
* `target_ids` → the same sequence shifted **one token to the right**, representing the **next token to predict**.
* `max_length` → number of tokens in each sequence.
* `stride` → how many tokens to move the sliding window each time.

**Example:**

```text
Tokens:  [1, 2, 3, 4, 5, 6]

Input:   [1, 2, 3, 4]
Target:  [2, 3, 4, 5]
```

The model learns:

```text
1 → 2
2 → 3
3 → 4
4 → 5
```

`stride` controls the overlap between consecutive sequences.

**In short:**

> Tokenized text → sliding windows → input/target pairs → PyTorch tensors → ready for LLM training.


In [21]:
import torch 
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, tokenizer, max_length,stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(data)
        for i in range(0,len(token_ids) - max_length,stride):
            input_seq = token_ids[i:i+max_length]
            target_seq = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_seq))
            self.target_ids.append(torch.tensor(target_seq))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## A data loader to generate batches with input-with pairs

In [22]:
from torch.utils.data import DataLoader

def create_dataloader_v1(data, batch_size = 32,
                         max_length =256,
                         stride = 128,
                         drop_last = True,
                         shuffle = True,
                         num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")  # Initializes the  tokenizer
    dataset = GPTDataset(data, tokenizer, max_length, stride)    # Creates dataset
    dataloader = DataLoader(dataset, 
                            batch_size=batch_size, 
                            shuffle=shuffle,
                            drop_last=drop_last,   # Drops the last batch if it is smaller than the specified batch size to prevent loss spike
                            num_workers=num_workers)
    return dataloader

lets test how our dataloader works 